# Paragraph boundary refinement

Develop candidate repairs on the training-side paragraph QA findings. Source bodies, release splits, original reference chunks and existing exclusions remain bound to their original snapshots. This stage proposes offset changes for independent review.

## Bind the private reference inputs

Use formal document bodies, the frozen program-only QA sample and the existing offset and exclusion ledgers.

In [1]:
import hashlib
import json
import os
import re
import tempfile
from bisect import bisect_left, bisect_right
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import nbformat
import pandas as pd
import pyarrow.parquet as pq
import tiktoken
from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import functions as F

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/12_reference_chunk_materialization.ipynb").is_file()
)
NOTEBOOK = ROOT / "notebooks/14_paragraph_boundary_refinement.ipynb"
n12 = nbformat.read(ROOT / "notebooks/12_reference_chunk_materialization.ipynb", as_version=4)
binder = next(cell.source for cell in n12.cells if cell.id == "12-bind")
reference = {"__name__": "__main__"}
exec(compile(binder, "12_reference_chunk_materialization.ipynb:12-bind", "exec"), reference)
spark = reference["spark"]
parents = reference["parents"]
qa_root = reference["qa_root"]
qa_manifest = reference["qa_manifest"]
audit = reference["audit"]
sha256_file = reference["sha256_file"]
original_offsets = reference["offsets"].select(
    "release_row", "ordinal", "start", "end", "chunk_sha256", "codepoints", "ml_split",
)
exact_holds = reference["excluded_occurrences"]
hold_ids = exact_holds.select("release_row").distinct()
reference_manifest = reference["output"] / "MANIFEST.json"
materialized = json.loads(reference_manifest.read_text(encoding="utf-8"))
if (
    materialized["offset_candidate_manifest_sha256"] != sha256_file(reference["qa_manifest_path"])
    or materialized["approved_for_model_input"]
):
    raise ValueError("Expected the unapproved, checksum-bound reference materialization.")

encoding_name = audit["reference_encoding"]
budget = audit["reference_content_token_budget"]
VENDOR_VOCAB = "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken"
VOCAB_SHA256 = "223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7"
cache_setting = os.environ.get("TIKTOKEN_CACHE_DIR", os.environ.get("DATA_GYM_CACHE_DIR"))
vocab_cache = Path(cache_setting) if cache_setting else Path(tempfile.gettempdir()) / "data-gym-cache"
vocabulary = vocab_cache / hashlib.sha1(VENDOR_VOCAB.encode()).hexdigest()
if encoding_name != "cl100k_base" or version("tiktoken") != "0.14.0" or (
    not vocabulary.is_file() or sha256_file(vocabulary) != VOCAB_SHA256
):
    raise ValueError("The pinned local reference vocabulary or tokenizer is unavailable.")
codec = tiktoken.get_encoding(encoding_name)

def canonical_sha256(value):
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


output_base = ROOT / ".cache/chunk-refinement"
output_base.mkdir(mode=0o700, exist_ok=True)
output = output_base / "paragraph-v2"


Spark 4.2.1 · local[6] · Notebook 06


Verified 718,330 parents and 12 handoff files.
Notebook 05 handoff manifest SHA-256: ff507ad49c929a8352669e93e46151371a64b280857010559aecb38c639bbfcf


,ml_split,overlap_status,count
0,test,direct_evidence,381
1,test,no_cross_split_evidence_in_current_audit,35424
2,test,related_release_group,1
3,train,direct_evidence,1029
4,train,no_cross_split_evidence_in_current_audit,645383
5,train,related_release_group,2
6,validation,direct_evidence,382
7,validation,no_cross_split_evidence_in_current_audit,35722
8,validation,related_release_group,6


## Register observed boundary failures

Validate each completed QA receipt. Develop proposals on training-side material findings. Count held-out findings separately without using their text for development.

In [2]:
# Validate the existing receipts before extracting train-side development cases.
n11 = nbformat.read(ROOT / "notebooks/11_program_chunk_qa.ipynb", as_version=4)
cell = next(c.source for c in n11.cells if c.id == "11-qa_status")
validation, terminator, _ = cell.partition("if len(completed) == sum(targets.values()):")
if not terminator:
    raise ValueError("The upstream QA receipt validator changed.")
qa = {
    "published": qa_manifest, "output": qa_root, "canonical_sha256": canonical_sha256,
    "Counter": Counter, "json": json, "targets": audit["fresh_sample_parents"],
}
exec(compile(validation, "11_program_chunk_qa.ipynb:11-qa_status", "exec"), qa)
reviews = qa["completed"]
if len(reviews) != sum(audit["fresh_sample_parents"].values()):
    raise ValueError("The boundary refinement requires the complete frozen QA sample.")
material_cases = [
    (packet, decision) for packet, decision in reviews
    if decision["boundary_outcome"] == "material_evidence_break"
]
by_split = Counter(packet["ml_split"] for packet, _ in material_cases)
if any(packet["boundary_kind"] != "paragraph_only" for packet, _ in material_cases):
    raise ValueError("A material finding has an unexpected reference boundary kind.")
training_rows = [
    (
        packet["release_row"], packet["clean_sha256"], packet["format"], packet["ends"],
        decision["affected_internal_ends"], decision["model_view_evidence_loss"],
        decision["packet_sha256"],
    )
    for packet, decision in material_cases if packet["ml_split"] == "train"
]
if len({row[0] for row in training_rows}) != len(training_rows):
    raise ValueError("Repeated parent in the training adjudication roster.")
cases = spark.createDataFrame(
    training_rows,
    "release_row long, clean_sha256 string, format string, baseline_ends array<int>, "
    "target_ends array<int>, exact_hold_evidence_loss boolean, packet_sha256 string",
)
identity = {
    "notebook_sources_sha256": canonical_sha256([
        (c.id, c.source) for c in nbformat.read(NOTEBOOK, as_version=4).cells
    ]),
    "reference_materialization_manifest_sha256": sha256_file(reference_manifest),
    "document_handoff_manifest_sha256": reference["identity"]["source_handoff_manifest_sha256"],
    "qa_manifest_sha256": sha256_file(reference["qa_manifest_path"]),
    "qa_receipts": {
        batch: sha256_file(qa_root / "decisions" / f"{batch}.jsonl")
        for batch in qa_manifest["qa_batches"]
    },
    "reference_vocabulary_sha256": VOCAB_SHA256,
}
print("Material findings by split", dict(by_split))
print(f"Bound {len(training_rows):,} training-side cases for proposal development.")


Semantic QA receipts verified: 1,500 of 1,500 parents.
Received findings: {'none': 580, 'boundary_preference': 330, 'material_evidence_break': 590}


Material findings by split {'train': 494, 'validation': 45, 'test': 51}
Bound 494 training-side cases for proposal development.


## Search for budget-safe paragraph alternatives

Try to keep neighboring paragraphs together at each reported edge. Use only recorded paragraph starts and count each proposed span with the pinned 512-content-token reference tokenizer. Preserve a route through the original plan and rank feasible trials by remaining reported edges, newly introduced context cues, boundary edits and then chunk count. Search up to the next 24 paragraph starts within 8,192 code points, with an exception for original valid spans. Retaining an edge means the bounded search did not find a complete repair under these settings.

In [3]:
SEARCH_ANCHORS = 24
MAX_NEW_SPAN_CODEPOINTS = 8192
DEPENDENT_OPENING = re.compile(
    r"^[\s\"'“‘(\[]*(?:this|that|these|those|it|they|he|she|however|"
    r"therefore|nevertheless|thus|instead|for example|in response|as a result)\b",
    re.IGNORECASE,
)

def cues_at_new_boundary(body, offset):
    left = body[max(0, offset - 96):offset].rstrip().rstrip(chr(34) + "'”’)]").rstrip()
    right = body[offset:offset + 96]
    return left.endswith(("?", ":")) or DEPENDENT_OPENING.search(right) is not None


def repartition_paragraphs(
    body, paragraph_starts, baseline_ends, target_ends, token_count, cap,
    search_anchors=SEARCH_ANCHORS, max_new_span=MAX_NEW_SPAN_CODEPOINTS,
):
    # The reference chunk sequence is a feasible fallback through the search graph.
    boundaries = sorted({0, len(body), *paragraph_starts})
    positions = {offset: index for index, offset in enumerate(boundaries)}
    baseline = set(baseline_ends)
    targets = set(target_ends)
    if (
        not body or baseline_ends != sorted(baseline) or baseline_ends[-1:] != [len(body)]
        or not baseline <= positions.keys() or 0 in baseline
        or not targets <= baseline - {len(body)}
    ):
        raise ValueError("The reviewed offsets lack a matching paragraph reference plan.")

    fallback = dict(zip([0, *baseline_ends[:-1]], baseline_ends, strict=True))
    for start, end in fallback.items():
        if not 0 < token_count(body[start:end]) <= cap:
            raise ValueError("A frozen reference span does not fit its recorded budget.")

    dependencies = {
        end for end in boundaries[1:-1]
        if end not in baseline and cues_at_new_boundary(body, end)
    }
    # After reducing known breaks, prefer plans with fewer newly flagged openings.
    costs = {0: (0, 0, 0, 0)}
    backtrace = {}
    for i, start in enumerate(boundaries[:-1]):
        if start not in costs:
            continue
        endpoints = boundaries[i + 1:i + 1 + search_anchors]
        if start in fallback and fallback[start] not in endpoints:
            endpoints.append(fallback[start])
        for end in endpoints:
            if end - start > max_new_span and end != fallback.get(start):
                continue
            count = token_count(body[start:end])
            if not 0 < count <= cap:
                continue
            old_edges_skipped = bisect_left(baseline_ends, end) - bisect_right(baseline_ends, start)
            missed, dependency_count, edits, chunks = costs[start]
            rank = (
                missed + int(end in targets),
                dependency_count + int(end in dependencies),
                edits + old_edges_skipped + int(end not in baseline),
                chunks + 1,
            )
            if end not in costs or rank < costs[end]:
                costs[end] = rank
                backtrace[end] = (start, count)

    end = len(body)
    new_ends = []
    counts = []
    while end:
        new_ends.append(end)
        start, count = backtrace[end]
        counts.append(count)
        end = start
    new_ends.reverse()
    counts.reverse()
    unresolved = sorted(targets & set(new_ends))
    dependency_ends = sorted(dependencies & set(new_ends))
    return new_ends, counts, unresolved, dependency_ends


def reference_token_count(text):
    return len(tiktoken.get_encoding(encoding_name).encode_ordinary(text))


test_parts = ["first words here\n\n", "second words here\n\n", "third words here\n\n"]
test_body = "".join(test_parts)
test_starts = [0, len(test_parts[0]), len(test_parts[0] + test_parts[1])]
test_reference = [*test_starts[1:], len(test_body)]
too_small_to_merge = max(reference_token_count(piece) for piece in test_parts)
enough_to_merge = reference_token_count(test_parts[0] + test_parts[1])
merged, merged_counts, unresolved, _ = repartition_paragraphs(
    test_body, test_starts, test_reference, [test_starts[1]],
    reference_token_count, enough_to_merge,
)
assert test_starts[1] not in merged and not unresolved and max(merged_counts) <= enough_to_merge
fixed, fixed_counts, unresolved, _ = repartition_paragraphs(
    test_body, test_starts, test_reference, [test_starts[1], test_starts[2]],
    reference_token_count, too_small_to_merge,
)
assert fixed == test_reference and unresolved == test_starts[1:] and max(fixed_counts) <= too_small_to_merge
preserved, _, _, _ = repartition_paragraphs(
    test_body, test_starts, test_reference, [], reference_token_count, enough_to_merge,
)
assert preserved == test_reference
assert cues_at_new_boundary("What happened?\n\nHeavy rain.", len("What happened?\n\n"))
assert cues_at_new_boundary(
    "The council rejected the proposal.\n\nThat is disputed.",
    len("The council rejected the proposal.\n\n"),
)
assert not cues_at_new_boundary("We left.\n\nHeavy rain followed.", 10)
print("Verified reference preservation, budget fallback and context-cue controls.")


Verified reference preservation, budget fallback and context-cue controls.


## Generate the training-side repair ledger

Keep parents with existing exact-held occurrences for explicit adjudication. Flag new cuts after questions or lead-ins and before demonstratives, pronouns or linking adverbs. A trial with a flagged new edge also needs explicit adjudication. Preserve partial proposals and unresolved offsets as diagnostics. Moving an edge is a geometric intervention. The QA report may describe evidence spanning more than two paragraphs, which needs paired semantic review.

In [4]:
search_udf = F.udf(
    lambda text, starts, ends, targets: repartition_paragraphs(
        text, starts, ends, targets, reference_token_count, budget
    ),
    "struct<ends:array<int>,token_counts:array<int>,unresolved:array<int>,dependencies:array<int>>",
    useArrow=False,
)
source_bodies = parents.where(F.col("ml_split") == "train").select(
    "release_row", "clean_sha256", "clean_text", "clean_chars",
    F.transform("paragraphs", lambda paragraph: paragraph.start).alias("paragraph_starts"),
)
scoped = source_bodies.join(F.broadcast(cases), ["release_row", "clean_sha256"])
blocked = (
    scoped.join(F.broadcast(hold_ids), "release_row")
    .select(*cases.columns)
    .withColumn("trial_ends", F.lit(None).cast("array<int>"))
    .withColumn("trial_token_counts", F.lit(None).cast("array<int>"))
    .withColumn("unresolved_ends", F.col("target_ends"))
    .withColumn("new_dependency_ends", F.lit(None).cast("array<int>"))
    .withColumn("proposal_status", F.lit("existing_exact_hold_requires_adjudication"))
)
searchable = scoped.join(F.broadcast(hold_ids), "release_row", "left_anti")
if searchable.where("exact_hold_evidence_loss").count():
    raise ValueError("An evidence-loss finding is missing its associated prior occurrence hold.")
trial = (
    searchable.withColumn(
        "search", search_udf("clean_text", "paragraph_starts", "baseline_ends", "target_ends")
    )
    .select(
        *cases.columns, F.col("search.ends").alias("trial_ends"),
        F.col("search.token_counts").alias("trial_token_counts"),
        F.col("search.unresolved").alias("unresolved_ends"),
        F.col("search.dependencies").alias("new_dependency_ends"),
    )
    .withColumn(
        "proposal_status",
        F.when(
            (F.size("unresolved_ends") == 0) & (F.size("new_dependency_ends") > 0),
            "new_context_cue_requires_adjudication",
        )
        .when(F.size("unresolved_ends") == 0, "all_reported_edges_removed")
        .when(F.size("unresolved_ends") < F.size("target_ends"), "some_reported_edges_removed")
        .otherwise("no_repair_under_search_constraints"),
    )
)
trials = trial.unionByName(blocked).persist(StorageLevel.DISK_ONLY)
if trials.count() != len(training_rows) or trials.select("release_row").distinct().count() != (
    len(training_rows)
):
    raise ValueError("The verified training body roster does not match the repair trials.")
trial_counts = trials.groupBy("proposal_status").agg(
    F.count("*").alias("parents"),
    F.sum(F.size("target_ends")).alias("reported_edges"),
    F.sum(F.size("unresolved_ends")).alias("unresolved_edges"),
    F.sum(F.size("new_dependency_ends")).alias("new_context_cues"),
).orderBy("proposal_status").toPandas()
display(trial_counts)


,proposal_status,parents,reported_edges,unresolved_edges,new_context_cues
0,all_reported_edges_removed,464,851,0,0.0
1,existing_exact_hold_requires_adjudication,1,1,1,NaN
2,new_context_cue_requires_adjudication,11,36,0,13.0
3,no_repair_under_search_constraints,6,10,10,0.0
4,some_reported_edges_removed,12,105,16,3.0


## Check the proposed geometry and exact reuse

Verify coverage, token budget, paragraph starts and the frozen reference plan. Flag fresh exact coincidences against active and previously held validation or test slices. Check the projected candidate again with flagged repairs excluded. The reference release and its existing occurrence holds retain their original identifiers.

In [5]:
complete_trials = trials.where(F.col("proposal_status") == "all_reported_edges_removed")
eligible_ids = complete_trials.select("release_row")
before = (
    original_offsets.join(F.broadcast(eligible_ids), "release_row")
    .groupBy("release_row")
    .agg(F.sort_array(F.collect_list("end")).alias("recorded_ends"))
)
if complete_trials.join(before, "release_row", "left").where(
    ~F.col("baseline_ends").eqNullSafe(F.col("recorded_ends"))
).count():
    raise ValueError("A proposed repair no longer matches its unheld reference plan.")
proposal_chunks = (
    complete_trials.join(
        searchable.select("release_row", "clean_text", "clean_chars", "paragraph_starts"),
        "release_row",
    )
    .select(
        "release_row", "trial_ends", "trial_token_counts", "clean_text", "clean_chars",
        "paragraph_starts", F.posexplode("trial_ends").alias("ordinal", "end"),
    )
    .withColumn("start", F.coalesce(F.get("trial_ends", F.col("ordinal") - 1), F.lit(0)))
    .withColumn("codepoints", F.col("end") - F.col("start"))
    .withColumn("reference_token_count", F.get("trial_token_counts", F.col("ordinal")))
    .withColumn(
        "chunk_sha256",
        F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256),
    )
    .withColumn("ml_split", F.lit("train"))
    .withColumn(
        "valid",
        (F.col("start") >= 0)
        & (F.col("codepoints") > 0)
        & (F.size("trial_ends") == F.size("trial_token_counts"))
        & (F.try_element_at("trial_ends", F.lit(-1)) == F.col("clean_chars"))
        & (
            (F.col("end") == F.col("clean_chars"))
            | F.array_contains("paragraph_starts", F.col("end"))
        )
        & F.col("reference_token_count").between(1, budget)
        & F.col("chunk_sha256").isNotNull(),
    )
    .drop("trial_ends", "trial_token_counts", "clean_text", "clean_chars", "paragraph_starts")
    .persist(StorageLevel.DISK_ONLY)
)
if proposal_chunks.where(~F.coalesce("valid", F.lit(False))).count():
    raise ValueError("A proposed offset sequence failed exact coverage or reference budget.")
sha_key = ["chunk_sha256", "codepoints"]
opposite_active = original_offsets.where(F.col("ml_split") != "train").select(*sha_key)
opposite_held = (
    exact_holds.join(parents.select("release_row", "ml_split"), "release_row")
    .where(F.col("ml_split") != "train")
    .withColumn("codepoints", F.col("end") - F.col("start"))
    .select(*sha_key)
)
conflicts = (
    opposite_active.unionByName(opposite_held)
    .join(F.broadcast(proposal_chunks.select(*sha_key).distinct()), sha_key, "left_semi")
    .distinct()
)
conflict_ids = (
    proposal_chunks.join(F.broadcast(conflicts), sha_key, "left_semi")
    .select("release_row").distinct()
)
reviewable = complete_trials.join(F.broadcast(conflict_ids), "release_row", "left_anti")
reviewable_ids = reviewable.select("release_row")
replacement_chunks = proposal_chunks.join(F.broadcast(reviewable_ids), "release_row", "left_semi")
projected = original_offsets.join(
    F.broadcast(reviewable_ids), "release_row", "left_anti"
).unionByName(replacement_chunks.select(*original_offsets.columns))
projected_exact_cross_split = (
    projected.groupBy(*sha_key)
    .agg(F.min("ml_split").alias("first_split"), F.max("ml_split").alias("last_split"))
    .where(F.col("first_split") != F.col("last_split")).count()
)
if projected_exact_cross_split:
    raise ValueError("Exact cross-split reuse survived the proposed-offset projection.")

ledger = (
    trials.join(
        F.broadcast(conflict_ids.withColumn("exact_conflict", F.lit(True))), "release_row", "left"
    )
    .na.fill(False, subset=["exact_conflict"])
    .withColumn(
        "disposition",
        F.when(F.col("exact_conflict"), "proposed_exact_conflict")
        .when(
            F.col("proposal_status") == "all_reported_edges_removed",
            "eligible_for_paired_semantic_review",
        )
        .otherwise(F.col("proposal_status")),
    )
)
gate_counts = {
    row.disposition: row["count"] for row in ledger.groupBy("disposition").count().collect()
}
summary = {
    "receipt_sample_documents": len(reviews),
    "material_parent_findings_by_split": dict(by_split),
    "train_cases": len(training_rows),
    "evaluation_split_cases_deferred": sum(n for k, n in by_split.items() if k != "train"),
    "train_reported_edges": sum(len(row[4]) for row in training_rows),
    "trial_status_counts": {
        row.proposal_status: int(row.parents) for row in trial_counts.itertuples(index=False)
    },
    "dispositions": gate_counts,
    "new_exact_conflict_hashes": conflicts.count(),
    "projected_exact_cross_split_groups": projected_exact_cross_split,
    "projected_replacement_chunks": replacement_chunks.count(),
    "adopted_parent_changes": 0,
    "search_max_paragraph_anchors_per_start": SEARCH_ANCHORS,
    "search_max_new_span_codepoints": MAX_NEW_SPAN_CODEPOINTS,
    "new_boundary_cue_rule": DEPENDENT_OPENING.pattern,
    "semantic_disposition": "paired_review_required",
    "reference_token_budget": budget,
}
display(pd.DataFrame(summary.items(), columns=["Measure", "Result"]))


,Measure,Result
0,receipt_sample_documents,1500
1,material_parent_findings_by_split,"{'train': 494, 'validation': 45, 'test': 51}"
2,train_cases,494
3,evaluation_split_cases_deferred,96
4,train_reported_edges,1003
5,trial_status_counts,"{'all_reported_edges_removed': 464, 'existing_..."
6,dispositions,"{'eligible_for_paired_semantic_review': 464, '..."
7,new_exact_conflict_hashes,0
8,projected_exact_cross_split_groups,0
9,projected_replacement_chunks,2193


## Freeze a private paired comparison

Freeze 32 development comparisons by hash order with hash-based presentation order. Reviewers should judge each version on retention of the claim, qualification, speaker or question and its supporting context. Also check for new material breaks. Confirm the result using both sides and record uncertain or unresolved cases separately. The matched review, model-specific tokenizer validation and release-level replay govern adoption.

In [6]:
target_comparisons = 32
if reviewable.count() < target_comparisons:
    raise ValueError("There are fewer candidate parents than the declared comparison cohort.")
review_draw = reviewable.withColumn(
    "draw_key",
    F.sha2(F.concat_ws("|", F.lit("paired-paragraph-repair-v1"), "packet_sha256"), 256),
).orderBy("draw_key", "release_row").limit(target_comparisons)
comparisons = review_draw.join(
    searchable.select("release_row", "clean_text", "clean_chars"), "release_row"
).orderBy("draw_key", "release_row")
if not output.exists():
    with tempfile.TemporaryDirectory(prefix=".paragraph-refinement-", dir=output_base) as tmp:
        stage = Path(tmp) / "snapshot"
        stage.mkdir(mode=0o700)
        ledger.write.mode("errorifexists").option("compression", "zstd").parquet(
            str(stage / "trial_ledger")
        )
        replacement_chunks.drop("valid").write.mode("errorifexists").option(
            "compression", "zstd"
        ).parquet(str(stage / "reviewable_proposed_offsets"))
        private = stage / "paired_review"
        private.mkdir(mode=0o700)
        packet_file = private / "PACKETS.jsonl"
        paired_count = 0
        with packet_file.open("x", encoding="utf-8") as handle:
            for row in comparisons.toLocalIterator():
                draw_key = int(row.draw_key[-2:], 16)
                sides = [row.baseline_ends, row.trial_ends]
                if draw_key % 2:
                    sides.reverse()
                packet = {
                    "release_row": row.release_row,
                    "clean_sha256": row.clean_sha256,
                    "clean_text": row.clean_text,
                    "format": row.format,
                    "reference_packet_sha256": row.packet_sha256,
                    "plan_a_ends": sides[0],
                    "plan_b_ends": sides[1],
                }
                if hashlib.sha256(row.clean_text.encode()).hexdigest() != row.clean_sha256:
                    raise ValueError("A comparison packet failed its body hash.")
                packet["comparison_sha256"] = canonical_sha256(packet)
                handle.write(json.dumps(packet, ensure_ascii=False) + "\n")
                paired_count += 1
        packet_file.chmod(0o400)
        if paired_count != target_comparisons:
            raise ValueError("The comparison packets do not cover the frozen draw.")
        if output.exists():
            raise FileExistsError("Another writer created the refinement snapshot.")
        receipt = {
            "identity": identity,
            "summary": {**summary, "paired_review_parents": paired_count},
            "files": [
                {"path": path.relative_to(stage).as_posix(), "sha256": sha256_file(path)}
                for path in sorted(stage.rglob("*"))
                if path.is_file() and not path.name.startswith(".")
            ],
        }
        (stage / "MANIFEST.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        stage.rename(output)

stored = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
if stored["identity"] != identity or stored["summary"] != {
    **summary, "paired_review_parents": target_comparisons
}:
    raise ValueError("The stored refinement differs from the current bound review run.")
for file in stored["files"]:
    if sha256_file(output / file["path"]) != file["sha256"]:
        raise ValueError("A sealed trial output failed verification.")
for name, expected in (
    ("trial_ledger", len(training_rows)), ("reviewable_proposed_offsets", replacement_chunks.count())
):
    if sum(
        pq.ParquetFile(path).metadata.num_rows for path in (output / name).glob("*.parquet")
    ) != expected:
        raise ValueError("The output ledger did not reconcile to the trial.")
print(f"Verified {target_comparisons} private paired comparisons for the next semantic gate.")
print(f"Local proposal snapshot: {output.relative_to(ROOT)}")
spark.stop()


Verified 32 private paired comparisons for the next semantic gate.
Local proposal snapshot: .cache/chunk-refinement/paragraph-v2
